<span style="color: blue">
<h1>Imported Packages</h1>
</span>


In [ ]:
import random
from collections import defaultdict
import math

<span style="color: blue">
<h1>Game Class and play_game Function</h1>
</span>


In [ ]:
# ============================================================
# Game: Abstract Base Class
# ============================================================
# This class defines the *interface* that any two-player, turn-taking game
# must implement to work with the search algorithms (minimax, alpha-beta,
# etc.). It doesn't play any specific game itself
# Why do it this way? It lets the same minimax/alpha-beta code work for
# ANY game (TicTacToe, Connect Four, etc.) without being rewritten
# the search algorithms only ever call these four methods, never anything
# game-specific.

class Game(object):
    """
    Abstract interface for a two-player game.

    """

    def actions(self, state):
        """
        Return the collection of legal moves available from `state`.

        For Connect Four, this will be "which columns aren't full yet" --
        a move is just a column index (0-6), since a piece always drops
        to the lowest open row in that column.
        """
        raise NotImplementedError  # forces subclasses to implement this -- calling it
                                    # on the base Game class directly is a bug, not
                                    # a valid "no moves" case

    def result(self, state, move):
        """
        Return the NEW state that results from playing `move` in `state`.
        
        """
        raise NotImplementedError

    def is_terminal(self, state):
        """
        Return True if the game has ended at `state` (win, loss, or draw).

        Default behavior: if there are no legal actions left, the game
        is over. This is a reasonable default for many games (including
        a full Connect Four board), but subclasses often override this
        to ALSO check for a win before the board is full.
        """
        return not self.actions(state)

    def utility(self, state, player):
        """
        Return the value of a TERMINAL state, from `player`'s point of view.

        Convention I will use: +1 = player wins, -1 = player loses, 0 = draw. 
        calling it mid-game is undefined.
        """
        raise NotImplementedError

# ============================================================
# play_game: Generic Game Loop
# ============================================================
# This function runs a full game from start to finish, alternating turns
# between whichever "strategies" (move-choosing functions) you give it.
# It doesn't know or care whether the strategies are minimax, alpha-beta,
# a random mover, or a human typing input it just calls whatever
# function is registered for the current player and asks it for a move.

def play_game(game: Game, strategies: dict, verbose=False):
    """
    Play one full game and return the final state.

    Parameters
    ----------
    game : Game
        An instance of a Game subclass (e.g. ConnectFour()). Provides
        the rules via actions/result/is_terminal/utility.
    strategies : dict
        Maps each player identifier -> a function(game, state) -> move.
        Example: {0: player(alphabeta_search), 1: player(random_player)}
        This is how you pit different bot difficulties (or a human)
        against each other without changing this function at all.
    verbose : bool
        If True, print each move and the resulting board as the game
        progresses -- useful for debugging and for visually sanity-checking
        that a difficulty tier is playing the way you expect.

    Returns
    -------
    The final game state, once game.is_terminal(state) is True. Call
    game.utility(final_state, player) afterward to see who won.
    """
    state = game.initial  # start from the game's defined starting position
                           # (e.g. an empty Connect Four board, player 0 to move)

    while not game.is_terminal(state):        # keep playing until someone wins,
                                                # loses, or the board fills up
        player = state.to_move                 # ask the current state whose turn it is
        move = strategies[player](game, state) # hand control to that player's strategy
                                                 # function -- this is where minimax /
                                                 # alpha-beta / random / human input
                                                 # actually gets invoked
        state = game.result(state, move)        # apply the chosen move and advance
                                                 # to the resulting new state

        if verbose:
            print('Player', player, 'move:', move)
            print(state)                        # relies on the state having a
                                                 # __repr__/__str__  we'll define
                                                 # that when we build the board
                                                 # representation

    return state  # the caller can inspect this to determine the winner,
                   # or feed it into game.utility() for a numeric result